# B1 - Resolution automatique de theoremes par SAT

Ce notebook complete le rapport du projet. Il presente le principe de l'encodage SAT, les trois familles de problemes implementees, les resultats du benchmark et les limites liees aux preuves UNSAT externes.

Le code source reproductible se trouve dans `src/`. Les resultats affiches ici correspondent aux fichiers generes dans `results/` par le script `run_experiments.py`.

## 1. Objectif du projet

L'objectif est de transformer des enonces combinatoires en formules booleennes CNF, puis de les resoudre avec des solveurs SAT. La CNF obtenue peut etre exportee au format DIMACS, ce qui rend l'experience independante du code Python.

Familles traitees:

- coloriage de graphe avec `k` couleurs;
- pigeonhole principle, c'est-a-dire injecter `n+1` pigeons dans `n` trous;
- Ramsey `R(3,3)`, via l'absence de triangle monochromatique dans une 2-coloration des aretes de `K_n`.

Solveurs compares:

- PySAT: `glucose3`, `cadical153`, `lingeling`;
- fallback pedagogique `dpll_fallback`;
- SMT Z3 sur une modelisation plus haut niveau.

## 2. Imports et verification des solveurs

La cellule suivante verifie la disponibilite des solveurs SAT utilises dans le benchmark. Elle documente la configuration experimentale.

In [ ]:
from pysat.solvers import Solver

for name in ["glucose3", "cadical153", "lingeling"]:
    with Solver(name=name, bootstrap_with=[[1]]) as solver:
        print(name, solver.solve())

Chaque ligne doit afficher `True`, ce qui signifie que le solveur est disponible et peut resoudre une CNF minimale.

## 3. Transformation de Tseitin

La transformation de Tseitin introduit une variable auxiliaire par sous-formule. Cela permet d'obtenir une CNF lineaire en la taille de la formule, contrairement a une distribution naive de `and` sur `or` qui peut exploser exponentiellement.

Exemple: on encode la contradiction `p and not p`. La formule doit etre UNSAT.

In [ ]:
from src.tseitin import And, Atom, Not, Or, to_cnf
from src.solver_sat import solve_cnf

contradiction = to_cnf(And([Atom("p"), Not(Atom("p"))])).cnf
print(contradiction.to_dimacs())
print(solve_cnf(contradiction, solver_name="glucose3").status)

Une formule satisfiable simple, `p or q`, donne au contraire un modele SAT.

In [ ]:
sat_formula = to_cnf(Or([Atom("p"), Atom("q")])).cnf
result = solve_cnf(sat_formula, solver_name="glucose3")
print(result.status)
print(result.model)

## 4. Encodages SAT implementes

### 4.1 Coloriage de graphe

Variable `x_v_c`: le sommet `v` prend la couleur `c`.

Contraintes:

- chaque sommet a au moins une couleur;
- chaque sommet a au plus une couleur;
- deux sommets adjacents ne partagent pas la meme couleur.

Dans le benchmark, on teste des cycles avec 2 couleurs: les cycles pairs sont SAT, les cycles impairs sont UNSAT.

### 4.2 Pigeonhole principle

Variable `x_p_h`: le pigeon `p` est place dans le trou `h`.

Contraintes:

- chaque pigeon est dans exactement un trou;
- deux pigeons ne peuvent pas partager le meme trou.

Les instances `n+1` pigeons vers `n` trous sont UNSAT.

### 4.3 Ramsey R(3,3)

Variable `e_u_v_red`: l'arete `(u, v)` est rouge. Faux signifie bleu.

Pour chaque triangle, on interdit le triangle tout rouge et le triangle tout bleu. `K_5` est SAT, tandis que `K_6` est UNSAT, ce qui illustre `R(3,3)=6`.

In [ ]:
from src.encodings import complete_graph, graph_coloring_instance, pigeonhole_instance, ramsey_instance
from src.validation import validate_model

examples = [
    graph_coloring_instance(3, complete_graph(3), 3, name="triangle_3color", expected_sat=True),
    graph_coloring_instance(3, complete_graph(3), 2, name="triangle_2color", expected_sat=False),
    pigeonhole_instance(3, 2, name="php_3_into_2"),
    ramsey_instance(5),
    ramsey_instance(6),
]

for instance in examples:
    result = solve_cnf(instance.cnf, solver_name="glucose3")
    print(instance.name, instance.family, instance.cnf.n_vars, len(instance.cnf.clauses), result.status)
    if result.satisfiable:
        print(validate_model(instance, result.model)["feasible"])


## 5. Instances du benchmark

| Famille | Instances | Attendu |
| :-- | :-- | :-- |
| Graph coloring | `C5`, `C6`, `C7`, `C8` avec 2 couleurs | cycles impairs UNSAT, cycles pairs SAT |
| Pigeonhole | `4->3`, `5->4`, `6->5` | UNSAT |
| Ramsey R(3,3) | `K4`, `K5`, `K6` | `K4`, `K5` SAT; `K6` UNSAT |

Chaque instance est resolue par `glucose3`, `cadical153`, `lingeling`, `dpll_fallback` et Z3.

## 6. Resultats du benchmark

Synthese issue de `results/benchmark_summary.csv`:

| Backend | Solveur | Famille | Instances | Disponibilite | Taux resolu | Exactitude | Temps moyen | Variables moy. | Clauses moy. |
| :-- | :-- | :-- | --: | --: | --: | --: | --: | --: | --: |
| SAT | cadical153 | graph coloring | 4 | 100% | 100% | 100% | 0.00055 s | 13.00 | 26.00 |
| SAT | cadical153 | pigeonhole | 3 | 100% | 100% | 100% | 0.00087 s | 20.67 | 83.33 |
| SAT | cadical153 | Ramsey R(3,3) | 3 | 100% | 100% | 100% | 0.00061 s | 10.33 | 22.67 |
| SAT | glucose3 | graph coloring | 4 | 100% | 100% | 100% | 0.00362 s | 13.00 | 26.00 |
| SAT | glucose3 | pigeonhole | 3 | 100% | 100% | 100% | 0.00023 s | 20.67 | 83.33 |
| SAT | glucose3 | Ramsey R(3,3) | 3 | 100% | 100% | 100% | 0.00012 s | 10.33 | 22.67 |
| SAT | lingeling | graph coloring | 4 | 100% | 100% | 100% | 0.00106 s | 13.00 | 26.00 |
| SAT | lingeling | pigeonhole | 3 | 100% | 100% | 100% | 0.00169 s | 20.67 | 83.33 |
| SAT | lingeling | Ramsey R(3,3) | 3 | 100% | 100% | 100% | 0.00155 s | 10.33 | 22.67 |
| SAT | dpll_fallback | graph coloring | 4 | 100% | 100% | 100% | 0.00012 s | 13.00 | 26.00 |
| SAT | dpll_fallback | pigeonhole | 3 | 100% | 100% | 100% | 0.00704 s | 20.67 | 83.33 |
| SAT | dpll_fallback | Ramsey R(3,3) | 3 | 100% | 100% | 100% | 0.00088 s | 10.33 | 22.67 |
| SMT | z3 | graph coloring | 4 | 100% | 100% | 100% | 0.01421 s | 13.00 | 26.00 |
| SMT | z3 | pigeonhole | 3 | 100% | 100% | 100% | 0.01035 s | 20.67 | 83.33 |
| SMT | z3 | Ramsey R(3,3) | 3 | 100% | 100% | 100% | 0.00416 s | 10.33 | 22.67 |

In [ ]:
import pandas as pd

raw = pd.read_csv("results/benchmark_raw.csv")
summary = pd.read_csv("results/benchmark_summary.csv")
summary

Figure generee par `run_experiments.py`:

![Benchmark SAT/SMT](benchmark_overview.png)

## 7. Verification des resultats

Les fichiers CSV confirment que:

- les 50 resolutions attendues sont presentes;
- aucun solveur n'est en `UNAVAILABLE`;
- tous les statuts sont `SATISFIABLE` ou `UNSATISFIABLE`;
- tous les resultats correspondent a l'attendu theorique;
- chaque modele SAT produit par un solveur SAT est reinterprete et valide par `src/validation.py`.

La cellule suivante reproduit ces controles.

In [ ]:
assert len(raw) == 50
assert len(summary) == 15
assert set(raw["status"]) <= {"SATISFIABLE", "UNSATISFIABLE"}
assert raw["expected_ok"].all()
sat_models = raw[(raw["backend"] == "SAT") & (raw["satisfiable"] == True)]
assert sat_models["valid_model"].all()
assert (summary[["available_rate", "solved_rate", "expected_ok_rate"]] == 1.0).all().all()
print("Benchmark coherence checks passed.")

## 8. SAT vs SMT: interpretation

L'approche SAT est plus bas niveau: elle demande d'ecrire explicitement les variables booleennes et les clauses. En retour, elle produit des CNF exportables en DIMACS et compatibles avec des solveurs CDCL efficaces.

L'approche SMT est plus directe pour exprimer certains problemes: le coloriage et le pigeonhole peuvent utiliser des variables entieres, des bornes et `Distinct`. Elle est donc plus lisible, mais moins proche du format de preuve SAT standard.

Sur ces petites instances, tous les solveurs repondent rapidement. Les temps ne doivent pas etre sur-interpretes: le but du benchmark est surtout de verifier la coherence des encodages et la disponibilite des solveurs.

## 9. Certificats UNSAT et DRAT

Le projet exporte une CNF UNSAT dans `results/proofs/php_4_into_3.cnf`. La verification DRAT externe depend de `drat-trim`, qui n'est pas une dependance Python et doit etre installe separement.

La recuperation directe d'une trace via PySAT depend du solveur et de la plateforme. Le script evite donc le mode `with_proof=True` par defaut et se limite a exporter la CNF UNSAT; une preuve externe peut ensuite etre verifiee avec `drat-trim`.

In [ ]:
from pathlib import Path

proof_cnf = Path("results/proofs/php_4_into_3.cnf")
print(proof_cnf.exists(), proof_cnf.stat().st_size if proof_cnf.exists() else 0)
print("Command to use if drat-trim and a DRAT proof are available:")
print("drat-trim instance.cnf proof.drat")

## 10. Conclusion

Le projet fournit une chaine SAT complete et reproductible: Tseitin, export DIMACS, encodages classiques, solveurs PySAT, comparaison SMT, validation de modeles et generation de resultats. Les resultats du benchmark sont coherents avec les proprietes theoriques attendues: cycles impairs non 2-coloriables, pigeonhole `n+1 -> n` insatisfiable, et `R(3,3)=6` via l'insatisfiabilite de `K_6` sans triangle monochromatique.